# Statistical vs ML feature extraction

Recomputes the Part 3 overlap from two catalogues (not a re-run of EDA or the selectors):

- Statistical FDR set (n = 20, time-at-risk excluded) from `eda.ipynb`
- ML consensus set (n = 13, union of within-model LOCO ∩ SHAP ∩ FFS top-20, PR-AUC) from `baseline_feature_selections.ipynb`

Writes figures and CSVs to `paper_results/03_stats_vs_ml/paper_figures/` and `code/analyzes/stats_vs_ml/paper_figures/`.

This is a **methods comparison** of two extraction procedures, not a biological finding. Jaccard is 5/28 ≈ 0.18.


In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle

HERE = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [HERE, *HERE.parents]
        if (p / "code" / "modeling" / "tools" / "figure_style.py").is_file()
    ),
    HERE,
)
sys.path.insert(0, str(ROOT / "code" / "modeling" / "tools"))
from figure_style import HARMONY, apply_style  # noqa: E402

OUT_DIRS = [
    ROOT / "paper_results" / "03_stats_vs_ml" / "paper_figures",
    ROOT / "code" / "analyzes" / "stats_vs_ml" / "paper_figures",
]
print("ROOT:", ROOT)


ROOT: /home/fadia/Documents/vlst


In [3]:
# FDR names from eda.ipynb (evidence map §8.1). Time-at-risk is tracked separately.
STATS_FDR = [
    "WBC",
    "eGFR",
    "LV",
    "CKD5",
    "No.of stents per lesion",
    "HbA1c",
    "NO.of vessels",
    "Total stent length",
    "Fiberinogen",
    "1.1:1Post dilation",
    "No postdilation",
    "CKD90",
    "Previous PCI",
    "3-vessel disease",
    "Clopidogrel",
    "Diabetes",
    "PES",
    "Multi-vessel CAD",
    "Single-vessel disease",
    "Stent type-SES",
]
# Union of within-model LOCO ∩ SHAP ∩ FFS (top-20, PR-AUC only) across 7 classic models.
# xgb's 7th names were truncated in the notebook HTML as "WB..."; the sorted list is
# 1.1:1Post dilation; Aneurysm; Cre; HGB; LV; WBC; eGFR (n_common=7).
ML_CONSENSUS = [
    "1.1:1Post dilation",
    "Aneurysm",
    "CaI",
    "Cre",
    "HGB",
    "HbA1c",
    "LDL",
    "LV",
    "LVEF",
    "Men",
    "UA",
    "WBC",
    "eGFR",
]
STATS_MULTIVAR = {
    "WBC",
    "eGFR",
    "LV",
    "CKD5",
    "1.1:1Post dilation",
    "CKD90",
    "Previous PCI",
    "Clopidogrel",
}
ML_FREQUENT_EXTRA = [
    "No postdilation",
    "Previous PCI",
    "STEMI",
    "stent overlap",
    "TCL",
    "TG",
    "CKD5",
    "Total stent length",
    "Clopidogrel",
    "Age",
    "Platelet",
    "Max-stent diameter",
]
TIME_AT_RISK = "Time since stent implantation"

DOMAINS = {
    "WBC": "Laboratory",
    "eGFR": "Laboratory",
    "LV": "Cardiac",
    "CKD5": "Laboratory / renal",
    "No.of stents per lesion": "Procedural",
    "HbA1c": "Laboratory",
    "NO.of vessels": "Anatomy",
    "Total stent length": "Procedural",
    "Fiberinogen": "Laboratory",
    "1.1:1Post dilation": "Procedural",
    "No postdilation": "Procedural",
    "CKD90": "Laboratory / renal",
    "Previous PCI": "History",
    "3-vessel disease": "Anatomy",
    "Clopidogrel": "Medication",
    "Diabetes": "Comorbidity",
    "PES": "Stent type",
    "Multi-vessel CAD": "Anatomy",
    "Single-vessel disease": "Anatomy",
    "Stent type-SES": "Stent type",
    "Cre": "Laboratory",
    "Men": "Demographics",
    "LVEF": "Cardiac",
    "HGB": "Laboratory",
    "Platelet": "Laboratory",
    "HL": "Comorbidity",
    "STEMI": "ACS presentation",
    "Hypertension": "Comorbidity",
    "Fast-Glu": "Laboratory",
    "TG": "Laboratory",
    "TCL": "Laboratory",
    "CaI": "Laboratory",
    "Min-stent diameter": "Procedural",
    "Current drinking": "Demographics",
    "History of HF": "History",
    TIME_AT_RISK: "Time-at-risk",
    "Chronic renal insufficiency": "Comorbidity",
    "HDL": "Laboratory",
    "History of peripheral vascular disease": "History",
    "Initial diagnosis-AMI": "ACS presentation",
    "LDL": "Laboratory",
    "NSTEMI": "ACS presentation",
    "Previous CABG": "History",
    "Previous MI": "History",
    "Stroke/TIA": "History",
    "UA": "ACS presentation",
    "Aneurysm": "Anatomy",
    "stent overlap": "Procedural",
    "Age": "Demographics",
    "Max-stent diameter": "Procedural",
}

SHARED_ROWS = [
    {
        "Feature": "WBC",
        "Domain": "Laboratory",
        "Statistical evidence": "MW r=0.13, q=9.5e-20",
        "ML evidence": "Global ML intersection; CatBoost/XGB/RF",
        "Why both methods keep it": "Mean shift and ranking feature (inflammation)",
    },
    {
        "Feature": "eGFR",
        "Domain": "Laboratory",
        "Statistical evidence": "Welch d=-0.71, q=3.7e-19",
        "ML evidence": "Global ML intersection; LOCO/SHAP core",
        "Why both methods keep it": "Renal filtration: models cannot compensate if dropped",
    },
    {
        "Feature": "LV",
        "Domain": "Cardiac",
        "Statistical evidence": "Welch d=1.13, q=3.3e-16",
        "ML evidence": "LOCO/SHAP cross-model; CatBoost/XGB",
        "Why both methods keep it": "Large location shift and a high-gain tree split",
    },
    {
        "Feature": "Fiberinogen",
        "Domain": "Laboratory",
        "Statistical evidence": "MW r=0.035, q=0.029",
        "ML evidence": "RF F2 consensus",
        "Why both methods keep it": "Weak haemostasis effect still used at recall-heavy F2",
    },
    {
        "Feature": "Previous PCI",
        "Domain": "History",
        "Statistical evidence": "Fisher OR=6.5, q=2e-4",
        "ML evidence": "RF F1 consensus",
        "Why both methods keep it": "Rare high-OR flag: 2x2 discovery and a pure binary split",
    },
]

STATS_ONLY_WHY = {
    "No postdilation": "Strong univariate OR; multivariable OR attenuates to ~1 after the complement is in the model. Boosting keeps the 1.1:1 flag instead",
    "CKD90": "Binary renal cutpoint; ML prefers continuous eGFR/Cre rather than the threshold",
    "CKD5": "FDR hit but collinear with eGFR; adjusted OR flips sign. Often selected, not in 3-way consensus",
    "3-vessel disease": "Anatomy binary; collinear with NO.of vessels / multi-vessel CAD",
    "Multi-vessel CAD": "Overlaps Single-vessel and 3-vessel; shared anatomical information",
    "Single-vessel disease": "Complement of multi-vessel disease (same 2×2 inverted)",
    "NO.of vessels": "Continuous vessel count; collinear with the vessel-disease binaries",
    "No.of stents per lesion": "Tiny univariate effect (MW r=0.037); not in any model three-way set",
    "Total stent length": "Small univariate effect; collinear with stent count / vessel burden",
    "Clopidogrel": "Full-cohort medication association; trees split on labs/procedure instead",
    "Diabetes": "Univariate FDR; multivariable CI includes 1; trees may split on HbA1c instead",
    "PES": "Stent-polymer binary; collinear with the 9-level brand column",
    "Stent type-SES": "χ² on 9 collapsed brands. ML one-hots those 9 levels (drop-first → 8 dummies); the parent name never enters a 3-way set",
    "Previous PCI": "Fisher OR=6.49. Frequently selected, but no model puts it in LOCO ∩ SHAP ∩ FFS on PR-AUC",
    "Fiberinogen": "Weak MW r=0.035. The old F2-consensus hit; PR-AUC three-way no longer keeps it",
}

ML_ONLY_WHY = {
    "Cre": ("ns (p=0.88)", "Univariate p=0.88 (redundant with eGFR). LR/XGB still use Cre as a renal surrogate"),
    "Men": ("ns (p=0.27)", "Univariate ns. Men×eGFR interaction is FDR-significant; LR uses sex as an additive offset"),
    "LVEF": ("raw p=0.033, FDR ns", "Borderline univariate (q=0.067). Joint logistic reverses sign (0.851 to 1.65) with LV in the model; trees still split on systolic function"),
    "HGB": ("raw p=0.039, FDR ns", "Raw p=0.039, FDR ns. CatBoost/RF/XGB three-way — ranking, not a location test"),
    "CaI": ("raw p=0.051, FDR ns", "Raw p=0.051, FDR ns. RF_b three-way; near the FDR boundary"),
    "LDL": ("ns (p=0.33)", "Univariate ns. RF three-way lipid split on the val-slice PR-AUC"),
    "UA": ("ns (p=0.17)", "Univariate ns. LR three-way ACS-presentation offset"),
    "Aneurysm": ("ns (p=0.40)", "Rare anatomy flag; XGB three-way only — treat as unstable"),
}

# One primary methodological bucket per compared name (Figure 3).
BUCKET = {
    "WBC": "Robust intersection",
    "eGFR": "Robust intersection",
    "LV": "Robust intersection",
    "HbA1c": "Robust intersection",
    "1.1:1Post dilation": "Robust intersection",
    "No postdilation": "Collinear family (stats-only)",
    "CKD90": "Collinear family (stats-only)",
    "CKD5": "Collinear family (stats-only)",
    "3-vessel disease": "Collinear family (stats-only)",
    "Multi-vessel CAD": "Collinear family (stats-only)",
    "Single-vessel disease": "Collinear family (stats-only)",
    "NO.of vessels": "Collinear family (stats-only)",
    "PES": "Collinear family (stats-only)",
    "Stent type-SES": "Brand encoding (9-level OHE, not parent name)",
    "No.of stents per lesion": "Weak for top-20 / not in three-way",
    "Total stent length": "Weak for top-20 / not in three-way",
    "Clopidogrel": "Weak for top-20 / not in three-way",
    "Diabetes": "Weak for top-20 / not in three-way",
    "Previous PCI": "Weak for top-20 / not in three-way",
    "Fiberinogen": "Weak for top-20 / not in three-way",
    "Cre": "Surrogate of an FDR hit",
    "Men": "Interaction / offset",
    "LVEF": "Borderline univariate, ML split",
    "HGB": "Borderline univariate, ML split",
    "CaI": "Borderline univariate, ML split",
    "LDL": "Hold-out / metric artefact",
    "UA": "Hold-out / metric artefact",
    "Aneurysm": "Hold-out / metric artefact",
    TIME_AT_RISK: "Structural time-at-risk",
}


In [4]:
def _save(fig, name: str) -> None:
    for out in OUT_DIRS:
        out.mkdir(parents=True, exist_ok=True)
        fig.savefig(out / name, dpi=300, bbox_inches="tight")


def _write_csv(df: pd.DataFrame, name: str) -> None:
    for out in OUT_DIRS:
        out.mkdir(parents=True, exist_ok=True)
        df.to_csv(out / name, index=False)


def _table_image(df: pd.DataFrame, name: str, col_widths: list[float] | None = None) -> None:
    n_rows, n_cols = df.shape
    fig_w = max(10, 0.22 * sum(len(str(c)) for c in df.columns) / 2)
    fig_h = max(2.2, 0.38 * (n_rows + 2))
    fig, ax = plt.subplots(figsize=(min(fig_w, 16), min(fig_h, 18)))
    ax.axis("off")
    tbl = ax.table(
        cellText=df.astype(str).values,
        colLabels=list(df.columns),
        loc="center",
        cellLoc="left",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1, 1.25)
    if col_widths:
        for i, w in enumerate(col_widths):
            for r in range(n_rows + 1):
                tbl[(r, i)].set_width(w)
    for (r, _), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor(HARMONY[7])
            cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#F4F7FA")
    _save(fig, name)
    plt.close(fig)


def build_membership() -> pd.DataFrame:
    stats, ml = set(STATS_FDR), set(ML_CONSENSUS)
    ml_freq = ml | set(ML_FREQUENT_EXTRA)
    rows = []
    for feat in sorted(stats & ml):
        rows.append(_member_row(feat, True, feat in STATS_MULTIVAR, True, True, "Both (FDR ∩ ML consensus)"))
    for feat in sorted(ml - stats):
        rows.append(_member_row(feat, False, False, True, True, "ML consensus only"))
    for feat in sorted(set(ML_FREQUENT_EXTRA) - stats - ml):
        rows.append(_member_row(feat, False, False, False, True, "ML frequent only"))
    for feat in sorted(stats - ml):
        rows.append(
            _member_row(
                feat,
                True,
                feat in STATS_MULTIVAR,
                False,
                feat in ml_freq,
                "Stats FDR only",
            )
        )
    rows.append(
        {
            "Feature": TIME_AT_RISK,
            "Stats FDR": "Yes",
            "Stats multivariable": "structural",
            "ML consensus": "No",
            "ML frequent": "No",
            "Set": "Structural (dropped from ML)",
        }
    )
    return pd.DataFrame(rows)


def _member_row(feat, fdr, multi, cons, freq, label):
    return {
        "Feature": feat,
        "Stats FDR": "Yes" if fdr else "No",
        "Stats multivariable": "Yes" if multi else "No",
        "ML consensus": "Yes" if cons else "No",
        "ML frequent": "Yes" if freq else "No",
        "Set": label,
    }


def fig_venn(stats: set[str], ml: set[str]) -> None:
    inter = stats & ml
    only_s, only_m = stats - ml, ml - stats
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    ax.set_aspect("equal")
    ax.axis("off")
    c1 = Circle((-0.55, 0), 1.05, facecolor=HARMONY[7], alpha=0.35, edgecolor=HARMONY[7], lw=2)
    c2 = Circle((0.55, 0), 1.05, facecolor=HARMONY[0], alpha=0.28, edgecolor=HARMONY[0], lw=2)
    ax.add_patch(c1)
    ax.add_patch(c2)
    ax.text(-1.15, 1.15, "Statistical FDR", fontsize=11, fontweight="bold", color=HARMONY[7], ha="center")
    ax.text(1.15, 1.15, "ML consensus", fontsize=11, fontweight="bold", color=HARMONY[0], ha="center")
    ax.text(-0.85, 0.15, f"n = {len(only_s)}", ha="center", fontsize=13, fontweight="bold")
    ax.text(0.85, 0.15, f"n = {len(only_m)}", ha="center", fontsize=13, fontweight="bold")
    ax.text(0, 0.08, f"n = {len(inter)}", ha="center", fontsize=13, fontweight="bold")
    ax.text(0, -0.28, "\n".join(sorted(inter)), ha="center", va="top", fontsize=8)
    union = len(stats | ml)
    jacc = len(inter) / union
    ax.set_xlim(-1.9, 1.9)
    ax.set_ylim(-1.45, 1.45)
    ax.set_title(
        f"Overlap of extraction catalogues   Jaccard = {jacc:.2f}  ({len(inter)} / {union})",
        pad=8,
    )
    _save(fig, "fig1_venn_overlap.png")
    plt.close(fig)


def fig_presence(member: pd.DataFrame) -> None:
    plot = member.copy()
    cols = ["Stats FDR", "Stats multivariable", "ML consensus", "ML frequent"]
    mat = []
    for _, r in plot.iterrows():
        row = []
        for c in cols:
            val = r[c]
            if val == "Yes":
                row.append(1)
            elif val == "structural":
                row.append(0.5)
            else:
                row.append(0)
        mat.append(row)
    mat = np.array(mat)
    fig_h = max(8, 0.28 * len(plot))
    fig, ax = plt.subplots(figsize=(7.2, fig_h))
    cmap = plt.cm.colors.ListedColormap(["#F4F7FA", HARMONY[8], HARMONY[7]])
    ax.imshow(mat, aspect="auto", cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(["Stats FDR", "Stats multivariable", "ML consensus", "ML frequent"], rotation=25, ha="right")
    ax.set_yticks(range(len(plot)))
    ax.set_yticklabels(plot["Feature"].tolist(), fontsize=7)
    ax.set_title("Feature presence by extractor")
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
    _save(fig, "fig2_presence_heatmap.png")
    plt.close(fig)


def fig_buckets() -> None:
    names = list(STATS_FDR) + [n for n in ML_CONSENSUS if n not in STATS_FDR] + [TIME_AT_RISK]
    counts = {}
    for n in names:
        b = BUCKET[n]
        counts[b] = counts.get(b, 0) + 1
    order = [
        "Robust intersection",
        "Collinear family (stats-only)",
        "Brand encoding (9-level OHE, not parent name)",
        "Weak for top-20 / not in three-way",
        "Surrogate of an FDR hit",
        "Interaction / offset",
        "Borderline univariate, ML split",
        "Hold-out / metric artefact",
        "Structural time-at-risk",
    ]
    order = [k for k in order if k in counts]
    vals = [counts[k] for k in order]
    colors = [HARMONY[i % len(HARMONY)] for i in range(len(order))]
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.barh(order[::-1], vals[::-1], color=colors[::-1])
    ax.set_xlabel("Number of compared names")
    ax.set_title("Primary methodological bucket (one bucket per feature)")
    for y, v in enumerate(vals[::-1]):
        ax.text(v + 0.15, y, str(v), va="center", fontsize=9)
    _save(fig, "fig3_reason_buckets.png")
    plt.close(fig)


def fig_domains(stats: set[str], ml: set[str]) -> None:
    domain_order = [
        "Laboratory",
        "Laboratory / renal",
        "Cardiac",
        "Procedural",
        "Anatomy",
        "Stent type",
        "History",
        "Comorbidity",
        "Medication",
        "Demographics",
        "ACS presentation",
    ]
    s_counts, m_counts = [], []
    for d in domain_order:
        s_counts.append(sum(1 for x in stats if DOMAINS.get(x) == d))
        m_counts.append(sum(1 for x in ml if DOMAINS.get(x) == d))
    keep = [i for i, (a, b) in enumerate(zip(s_counts, m_counts)) if a or b]
    labels = [domain_order[i] for i in keep]
    s_counts = [s_counts[i] for i in keep]
    m_counts = [m_counts[i] for i in keep]
    y = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(8.2, 5.4))
    ax.barh(y - 0.18, s_counts, height=0.35, color=HARMONY[7], label="Statistical FDR")
    ax.barh(y + 0.18, m_counts, height=0.35, color=HARMONY[0], label="ML consensus")
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Count of extracted names")
    ax.set_title("Extracted features by clinical domain")
    ax.legend(frameon=False, loc="lower right")
    _save(fig, "fig4_domain_counts.png")
    plt.close(fig)


In [5]:
apply_style()
stats, ml = set(STATS_FDR), set(ML_CONSENSUS)
inter = stats & ml
union = stats | ml
assert len(STATS_FDR) == 20
assert len(ML_CONSENSUS) == 13
assert inter == {"WBC", "eGFR", "LV", "HbA1c", "1.1:1Post dilation"}
assert len(inter) == 5
assert len(union) == 28
jacc = len(inter) / len(union)
print(f"stats={len(stats)} ml={len(ml)} intersection={len(inter)} union={len(union)} Jaccard={jacc:.4f}")

member = build_membership()
_write_csv(member, "table_feature_by_method.csv")
_table_image(member, "table_feature_by_method.png")

shared = pd.DataFrame(SHARED_ROWS)
_write_csv(shared, "table_shared_features.csv")
_table_image(shared, "table_shared_features.png")

stats_only = pd.DataFrame(
    [
        {
            "Feature": f,
            "Domain": DOMAINS[f],
            "Why statistical methods keep it and ML top-20 does not": STATS_ONLY_WHY[f],
        }
        for f in STATS_FDR
        if f not in ml
    ]
)
_write_csv(stats_only, "table_stats_only.csv")
_table_image(stats_only, "table_stats_only.png")

ml_only = pd.DataFrame(
    [
        {
            "Feature": f,
            "Univariate vs VLST": ML_ONLY_WHY[f][0],
            "Why ML consensus keeps it and FDR does not": ML_ONLY_WHY[f][1],
        }
        for f in ML_CONSENSUS
        if f not in stats
    ]
)
_write_csv(ml_only, "table_ml_only.csv")
_table_image(ml_only, "table_ml_only.png")

fig_venn(stats, ml)
fig_presence(member)
fig_buckets()
fig_domains(stats, ml)
print("Wrote figures and tables to:")
for d in OUT_DIRS:
    print(" ", d)


stats=20 ml=13 intersection=5 union=28 Jaccard=0.1786
Wrote figures and tables to:
  /home/fadia/Documents/vlst/paper_results/03_stats_vs_ml/paper_figures
  /home/fadia/Documents/vlst/code/analyzes/stats_vs_ml/paper_figures
